# Used Car Price Prediction

The goal of this competition is to develop a predictive model that estimates the price of used cars based on various attributes such as brand, mileage, fuel type, engine specifications, and more. By leveraging these features, we aim to provide accurate price predictions that can help both buyers and sellers make informed decisions in the used car market.

You can download the dataset for this competition from [here](https://www.kaggle.com/competitions/playground-series-s4e9/data).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error



In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.isnull().sum()

id                  0
brand               0
model               0
model_year          0
milage              0
fuel_type        5083
engine              0
transmission        0
ext_col             0
int_col             0
accident         2452
clean_title     21419
price               0
dtype: int64

In [4]:
test.isnull().sum()

id                  0
brand               0
model               0
model_year          0
milage              0
fuel_type        3383
engine              0
transmission        0
ext_col             0
int_col             0
accident         1632
clean_title     14239
dtype: int64

In [5]:
train['fuel_type'].unique()

array(['Gasoline', 'E85 Flex Fuel', nan, 'Hybrid', 'Diesel',
       'Plug-In Hybrid', '–', 'not supported'], dtype=object)

In [6]:
train["fuel_type"].replace({np.nan:"none",'–':'none','not supported':'none'}, inplace=True)
test["fuel_type"].replace({np.nan:"none",'–':'none','not supported':'none'}, inplace=True)

In [7]:
train["fuel_type"] = train["fuel_type"].replace({"E85 Flex Fuel":"Flex Fuel"})
test["fuel_type"] = test["fuel_type"].replace({"E85 Flex Fuel":"Flex Fuel"})

In [8]:
train['accident'].fillna('None reported',inplace=True)
test['accident'].fillna('None reported',inplace=True)

In [9]:
train['clean_title'].fillna('No',inplace=True)
test['clean_title'].fillna('No',inplace=True)

In [10]:
train["engine"].replace({np.nan:"none",'–':'none',"...":'none'}, inplace=True)
test["engine"].replace({np.nan:"none",'–':'none',"...":'none'}, inplace=True)

In [11]:
# Top 5 values
col = ['brand','model','int_col','ext_col','model_year','fuel_type','transmission','engine','fuel_type','clean_title','accident']
for c in col:
    print(train[c].value_counts()[:5],end='\n\n')

brand
Ford             23088
Mercedes-Benz    19172
BMW              17028
Chevrolet        16335
Audi             10887
Name: count, dtype: int64

model
F-150 XLT             2945
M3 Base               2229
Camaro 2SS            1709
M4 Base               1622
Mustang GT Premium    1526
Name: count, dtype: int64

int_col
Black    107674
Beige     24495
Gray      21204
Brown      5810
Red        5145
Name: count, dtype: int64

ext_col
Black     48658
White     43815
Gray      25293
Silver    16995
Blue      14555
Name: count, dtype: int64

model_year
2021    18198
2018    16414
2020    15848
2022    15749
2019    15409
Name: count, dtype: int64

fuel_type
Gasoline     165940
Hybrid         6832
none           5879
Flex Fuel      5406
Diesel         3955
Name: count, dtype: int64

transmission
A/T                               49904
8-Speed A/T                       20645
Transmission w/Dual Shift Mode    19255
6-Speed A/T                       18044
6-Speed M/T                       11

In [12]:
train_idx = train.loc[train["fuel_type"]=="none","engine"].index
test_idx = test.loc[test["fuel_type"]=="none","engine"].index

In [13]:
def impute_fuel_type(fuel_type, engine):
    if(fuel_type=="none"):
        pattern = r'\b(Flex|Diesel|Plug-In|Hybrid|Electric)\b'
        matches = re.findall(pattern, engine, re.IGNORECASE)
        matches = [m.lower() for m in matches]

        if "flex" in matches:
            return "Flex Fuel"
        elif "diesel" in matches:
            return "Diesel"
        elif "plug-in" in matches:
            return "Plug-In Hybrid"
        elif "hybrid" in matches:
            return "Hybrid"
        elif "electric" in matches:
            return "Electric"
        else:
            return "Gasoline"
        
    else:
        return fuel_type


train["fuel_type"] = train.apply(lambda row: impute_fuel_type(row["fuel_type"], row["engine"]),axis=1)

test["fuel_type"] = test.apply(lambda row: impute_fuel_type(row["fuel_type"], row["engine"]),axis=1)

In [14]:
def extract_engine(s: str):

    # Extract horsepower
    hpgroup = re.search(r'(\d+(\.\d+)?)\s*hp', s, re.IGNORECASE)
    engine_hp = float(hpgroup.group(1)) if hpgroup else np.nan
    
    # Extract liters (L or Liter)
    litergroup = re.search(r'(\d+(\.\d+)?)\s*[Ll](?:iter)?', s, re.IGNORECASE)
    engine_liter = float(litergroup.group(1)) if litergroup else np.nan
    
    # Extract cylinder count
    cylindergroup = re.search(r'(\d+)\s*cylinders?|([ivxlcdm]+)\s*cylinders?', s, re.IGNORECASE)
    if cylindergroup:
        engine_cyl = int(cylindergroup.group(1)) if cylindergroup.group(1) else cylindergroup.group(2).upper()
    else:
        engine_cyl = np.nan
    
    
    return engine_hp, engine_liter, engine_cyl
 

train[['engine_hp','engine_liter','engine_cyl']]=train['engine'].apply(extract_engine).apply(pd.Series)

test[['engine_hp','engine_liter','engine_cyl']]=test['engine'].apply(extract_engine).apply(pd.Series)

In [15]:
engine_hp_mode = train["engine_hp"].mode()[0]
train["engine_hp"].fillna(engine_hp_mode, inplace=True)

engine_liter_mode = train["engine_liter"].mode()[0]
train["engine_liter"].fillna(engine_liter_mode, inplace=True)

engine_cyl_mode = train["engine_cyl"].mode()[0]
train["engine_cyl"].fillna(engine_cyl_mode, inplace=True)

In [16]:
engine_hp_mode = test["engine_hp"].mode()[0]
test["engine_hp"].fillna(engine_hp_mode, inplace=True)

engine_liter_mode = test["engine_liter"].mode()[0]
test["engine_liter"].fillna(engine_liter_mode, inplace=True)

engine_cyl_mode = test["engine_cyl"].mode()[0]
test["engine_cyl"].fillna(engine_cyl_mode, inplace=True)

In [17]:
def extract_transmission(s:str):    
    # extract speed
    speedgroup = re.search(r'(\d+)', s)
    speed = int(speedgroup.group(1)) if speedgroup else np.nan
    
    # search for automatic
    if re.search(r'\b((a\/t)|automatic|at)\b', s, re.IGNORECASE):
        gear = 'Automatic'
    elif re.search(r'\b((m\/t)|manual|mt)\b', s, re.IGNORECASE):
        gear = 'Manual'
    elif re.search(r'\b((at\/mt)|dual)\b', s, re.IGNORECASE):
        gear = 'Dual'
    elif re.search(r'\b(cvt)\b', s, re.IGNORECASE):
        gear = 'CVT'
    else:
        gear = 'other'
    
    return speed, gear

# apply the function
train[['speed', 'gear']] = train['transmission'].apply(extract_transmission).apply(pd.Series)

test[['speed', 'gear']] = test['transmission'].apply(extract_transmission).apply(pd.Series)

In [18]:
speed_mode = train["speed"].mode()[0]
train["speed"].fillna(speed_mode, inplace=True)

gear_mode = train["gear"].mode()[0]
train["gear"].fillna(gear_mode, inplace=True)

# test dataset
speed_mode = test["speed"].mode()[0]
test["speed"].fillna(speed_mode, inplace=True)

gear_mode = test["gear"].mode()[0]
test["gear"].fillna(gear_mode, inplace=True)

In [19]:
train["car_old"] = 2024 - train["model_year"]

test["car_old"] = 2024 - test["model_year"]

In [20]:
def transform_basecolor(s: str):
    s = s.lower()
    base_colors = ['white', 'black', 'grey', 'gray', 'blue', 'red', 'yellow', 'silver',
                   'green', 'beige', 'gold', 'orange', 'brown', 'ebony', 'purple']
    
    # Find the first matching base color
    color = next((i for i in base_colors if i in s), 'uncommon')
    
    # Replace 'grey' with 'gray'
    if color == 'grey': 
        color = 'gray'
    
    return color


In [21]:
train['ext_col']=train['ext_col'].apply(transform_basecolor)
train['int_col']=train['int_col'].apply(transform_basecolor)
   
test['ext_col']=test['ext_col'].apply(transform_basecolor)
test['int_col']=test['int_col'].apply(transform_basecolor)

In [22]:
y= train["price"]
x = train[['brand', 'milage', 'fuel_type', 'ext_col', 'int_col', 'accident', 'clean_title',
                               'engine_hp', 'engine_liter', 'engine_cyl', 'speed', 'gear','car_old']]
test= test[['brand', 'milage', 'fuel_type', 'ext_col', 'int_col', 'accident', 'clean_title',
                               'engine_hp', 'engine_liter', 'engine_cyl', 'speed', 'gear','car_old']]

In [27]:
X = pd.get_dummies(x, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# Align the test dataset with training data columns (in case some columns are missing in the test data)
x, test = X.align(test, join='left', axis=1, fill_value=0)

# Create a pipeline with StandardScaler and Linear Regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),   # Scaling the features
    ('regressor', LinearRegression())  # Linear Regression model
])

# Fit the pipeline on training data
pipeline.fit(x, y)

# Predict on the test data
y_pred = pipeline.predict(test)

# Prepare the submission DataFrame
submission = pd.DataFrame({
    'id': range(1, len(test) + 1),  # Create a simple range for IDs
    'PredictedPrice': y_pred
})
# Save the predictions in a CSV file
submission.to_csv('submission.csv', index=False)

# Optionally, you can evaluate the model performance on the training data
y_train_pred = pipeline.predict(X)
mse = mean_squared_error(y, y_train_pred)
print(f"Mean Squared Error on Training Data: {mse}")

Mean Squared Error on Training Data: 5469187908.343058


### Conclusion

The linear regression model achieved a mean squared error (MSE) of **5.47 billion** on the training data. This indicates potential for improvement, and further model optimization or feature engineering may enhance performance.
